In [1]:
### /env/dependencies/libROM/build/examples/prom# ./elliptic_eigenproblem_global_rom -online -p 2 -rs 2 -n 4 -a 0.5 -ef 1.0 "-neu"

In [2]:
import os
import io
import sys
import time
try:
    import mfem.par as mfem
except ModuleNotFoundError:
    msg = "PyMFEM is not installed yet. Install PyMFEM:\n"
    msg += "\tgit clone https://github.com/mfem/PyMFEM.git\n"
    msg += "\tcd PyMFEM\n"
    msg += "\tpython3 setup.py install --with-parallel\n"
    raise ModuleNotFoundError(msg)

import ctypes
from ctypes import c_double
from mfem.par import intArray
from os.path import expanduser, join, dirname
import numpy as np
from numpy import sin, cos, exp, sqrt, pi, abs, array, floor, log, sum
from pylibROM.python_utils.StopWatch import StopWatch

In [3]:
sys.path.append("../../build")
import pylibROM.linalg as libROM
from pylibROM.mfem import ComputeCtAB

In [4]:
from mpi4py import MPI
comm = MPI.COMM_WORLD
myid = comm.Get_rank()
num_procs = comm.Get_size()

In [5]:
from mfem.common.arg_parser import ArgParser

# Create the parser
parser = ArgParser(description='elliptic_eigenproblem_global_rom')

In [6]:
# Default values
# mesh_file = "/notebooks/star.mesh"
mesh_file = ""
ser_ref_levels = 2
par_ref_levels = 1
dirichlet = True
order = 1
nev = 4
seed = 75
prescribe_init = False
lobpcg_niter = 200
lobpcg_tol = 1e-8
eig_tol = 1e-6
slu_solver = False
sp_solver = False
cpardiso_solver = False

visualization = True
visit = False
vis_steps = 5
fom = False
offline = False
merge = False
online = False
id = 0
nsets = 0
ef = 1.0
rdim = -1
verbose_level = 1

precision = 8

problem = 1
amplitude = 0
relative_center = 0.0
L = 1.0 # Scaling Factor
potential_well_switch = 0
psedo_time = 0.0


# Add arguments
parser.add_argument('-m', '--mesh', type=str, default=mesh_file, help='Mesh file to use.')
parser.add_argument('-p', '--problem', type=int, default=problem, help='Problem setup to use.')
# parser.add_argument('-dir', '--dirichlet', action='store_true', help='BC switch.')
# parser.add_argument('-neu', '--neumann', action='store_false', help='BC switch.')

# Add the Dirichlet argument with a default of True
parser.add_argument('-dir', '--dirichlet', action='store_false', default=True, help='Use Dirichlet boundary condition (default: False).')
# Add the Neumann argument with a default of False
parser.add_argument('-neu', '--neumann', action='store_true', default=False, help='Use Neumann boundary condition (default: True).')

parser.add_argument('-rs', '--refine-serial', type=int, default=ser_ref_levels, help='Number of times to refine the mesh uniformly in serial.')
parser.add_argument('-rp', '--refine-parallel', type=int, default=par_ref_levels, help='Number of times to refine the mesh uniformly in parallel.')
parser.add_argument('-o', '--order', type=int, default=order, help='Order (degree) of the finite elements.')
parser.add_argument('-n', '--num-eigs', type=int, default=nev, help='Number of desired eigenmodes.')
parser.add_argument('-s', '--seed', type=int, default=seed, help='Random seed used to initialize LOBPCG.')
parser.add_argument('-a', '--amplitude', type=float, default=amplitude, help='Amplitude of coefficient fields.')
parser.add_argument('-t', '--pseudo-time', type=float, default=psedo_time, help='Pseudo-time of the motion of the center.')
parser.add_argument('-c', '--center', type=float, default=relative_center, help='Number of grid elements to which center is shifted.')
parser.add_argument('-id', '--id', type=int, default=id, help='Parametric id.')
parser.add_argument('-ns', '--nset', type=int, default=nsets, help='Number of parametric snapshot sets.')
parser.add_argument('-vis', '--visualization', action='store_true', help='Enable GLVis visualization.')
parser.add_argument('-visit', '--visit-datafiles', action='store_true', help='Save data files for VisIt visualization.')
parser.add_argument('-vs', '--visualization-steps', type=int, default=vis_steps, help='Visualize every n-th timestep.')
parser.add_argument('-fom', '--fom', action='store_true', help='Enable the fom phase.')
parser.add_argument('-offline', '--offline', action='store_true', help='Enable the offline phase.')
parser.add_argument('-online', '--online', action='store_true', help='Enable the online phase.')
parser.add_argument('-merge', '--merge', action='store_true', help='Enable the merge phase.')
parser.add_argument('-ef', '--energy_fraction', type=float, default=ef, help='Energy fraction.')
parser.add_argument('-rdim', '--rdim', type=int, default=rdim, help='Reduced dimension.')
parser.add_argument('-v', '--verbose', type=int, default=verbose_level, help='Set the verbosity level of the LOBPCG solver and preconditioner.')
parser.add_argument('-fi', '--fom-iter', type=int, default=lobpcg_niter, help='Number of iterations for the LOBPCG solver.')
parser.add_argument('-tol', '--fom-tol', type=float, default=lobpcg_tol, help='Tolerance for the LOBPCG solver.')
parser.add_argument('-etol', '--eig-tol', type=float, default=eig_tol, help='Tolerance for eigenvalues to be considered equal.')

parser.add_argument("-d", "--device",
                    action='store', default='cpu', type=str,
                    help="Device configuration string, see Device::Configure().")

parser.add_argument('-slu', '--superlu', default=slu_solver, help='Use the SuperLU Solver.')
parser.add_argument('-sp', '--strumpack', default=sp_solver, help='Use the STRUMPACK Solver.')
parser.add_argument('-cpardiso', '--cpardiso', default = cpardiso_solver, help='Use the MKL CPardiso Solver.')



# Set precision
import sys
sys.stdout.write(f"{precision:.{precision}f}\n")

8.00000000


11

In [7]:
## run the offline phase 2 times to generate 2 snapshots
# args = parser.parse_args(["-offline", "-p", "2", "-rs", "2", "-n", "4","-id", "0", "-a", "0","-neu"])
# args = parser.parse_args(["-offline", "-p", "2", "-rs", "2", "-n", "4","-id", "1", "-a", "1","-neu"])


# args = parser.parse_args(["-merge", "-p", "2", "-rs", "2", "-n", "4", "-ns", "2", "-neu"])

# args = parser.parse_args(["-fom", "-p", "2", "-rs", "2", "-n", "4", "-a", "0.5","-neu"])


# args = parser.parse_args(["-online", "-p", "2", "-rs", "2", "-n", "4", "-a", "0.5", "-ef", "1.0", "-neu"])


In [8]:
## run the offline phase 2 times to generate 2 snapshots
args = parser.parse_args(["-offline", "-p", "2", "-rs", "2", "-n", "4","-id", "0", "-a", "0"])
# args = parser.parse_args(["-offline", "-p", "2", "-rs", "2", "-n", "4","-id", "1", "-a", "1"])


# args = parser.parse_args(["-merge", "-p", "2", "-rs", "2", "-n", "4", "-ns", "2"])

# args = parser.parse_args(["-fom", "-p", "2", "-rs", "2", "-n", "4", "-a", "0.5"])


# args = parser.parse_args(["-online", "-p", "2", "-rs", "2", "-n", "4", "-a", "0.5", "-ef", "1.0"])

In [9]:
# Parse the arguments
# args = parser.parse_args(["-p 2 -rs 2 -n 4 -id 0 -a 0"])
if (myid == 0): 
    parser.print_options(args)

Options used:
   --mesh  
   --problem  2
   --dirichlet  True
   --neumann  False
   --refine_serial  2
   --refine_parallel  1
   --order  1
   --num_eigs  4
   --seed  75
   --amplitude  0.0
   --pseudo_time  0.0
   --center  0.0
   --id  0
   --nset  0
   --visualization  False
   --visit_datafiles  False
   --visualization_steps  5
   --fom  False
   --offline  True
   --online  False
   --merge  False
   --energy_fraction  1.0
   --rdim  -1
   --verbose  1
   --fom_iter  200
   --fom_tol  1e-08
   --eig_tol  1e-06
   --device  cpu
   --superlu  False
   --strumpack  False
   --cpardiso  False


In [10]:
mesh_file = args.mesh
ser_ref_levels = args.refine_serial
par_ref_levels = args.refine_parallel
dirichlet = args.dirichlet
neumann = args.neumann
order = args.order
nev = args.num_eigs
seed = args.seed
slu_solver = args.superlu
sp_solver = args.strumpack
cpardiso_solver = args.cpardiso

visualization = args.visualization
visit = args.visit_datafiles
vis_steps = args.visualization_steps
fom = args.fom
offline = args.offline
merge = args.merge
online = args.online
id = args.id
nsets = args.nset
ef = args.energy_fraction
rdim = args.rdim
verbose_level = args.verbose
problem = args.problem
amplitude = args.amplitude
relative_center = args.center



# Check for conflicting solvers
if args.superlu and args.strumpack:
    print("WARNING: Both SuperLU and STRUMPACK have been selected, please choose either one.")
    print("         Defaulting to SuperLU.")
    args.strumpack = False


In [11]:
if (fom):
    if (not (fom and (not offline) and (not online))):
        raise ValueError("offline and online must be turned off if fom is used.")
else:
    check = (offline and (not merge) and (not online))          \
            or ((not offline) and merge and (not online))       \
            or ((not offline) and (not merge) and online)
    if (not check):
        raise ValueError("only one of offline, merge, or online must be true!")

In [12]:
# 3. Enable hardware devices such as GPUs, and programming models such as
#    CUDA, OCCA, RAJA and OpenMP based on command line options.
device = mfem.Device(args.device)
if (myid == 0):
    device.Print()

Device configuration: cpu
Memory configuration: host-std


In [13]:
# 4. Read the (serial) mesh from the given mesh file on all processors.  We
#    can handle triangular, quadrilateral, tetrahedral, hexahedral, surface
#    and volume meshes with the same code.

# Create a mesh
if args.mesh == "":
    mesh = mfem.Mesh(mfem.Mesh.MakeCartesian2D(2, 2, mfem.Element.QUADRILATERAL))
else:
    mesh = mfem.Mesh(args.mesh, 1, 1)

dim = mesh.Dimension()

In [14]:
# Get the bounding box
bb_min, bb_max = mesh.GetBoundingBox(max(args.order, 1))

In [15]:
# Vh = mfem.Vector()
# Vk z= mfem.Vector()

h_min = mfem.doublep(); 
h_max = mfem.doublep(); 
kappa_min = mfem.doublep(); 
kappa_max = mfem.doublep()

mesh.GetCharacteristics(h_min, h_max, kappa_min, kappa_max)

h_min = h_min.value();
h_max = h_max.value(); 
kappa_min = kappa_min.value(); 
kappa_max = kappa_max.value()

In [16]:
# Update h_max
h_max *= np.power(0.5, ser_ref_levels + par_ref_levels)

# Calculate the center
center = mfem.Vector(dim)
for i in range(dim):
    center[i] = h_max * relative_center + 0.5 * (bb_min[i] + bb_max[i])

In [17]:
# 5. Refine the serial mesh on all processors to increase the resolution. In
#    this example we do 'ref_levels' of uniform refinement. We choose
#    'ref_levels' to be the largest number that gives a final mesh with no
#    more than 10,000 elements.
for l in range(ser_ref_levels):
    mesh.UniformRefinement()

In [18]:
# 6. Define a parallel mesh by a partitioning of the serial mesh. Refine
#    this mesh further in parallel to increase the resolution. Once the
#    parallel mesh is defined, the serial mesh can be deleted.
pmesh = mfem.ParMesh(comm, mesh)
mesh.Clear()
for l in range(par_ref_levels):
    pmesh.UniformRefinement()

In [19]:
# 7. Define a parallel finite element space on the parallel mesh. Here we
#    use continuous Lagrange finite elements of the specified order. If
#    order < 1, we instead use an isoparametric/isogeometric space.
if (order > 0):
    fec = mfem.H1_FECollection(order, dim)
    delete_fec = True
elif (pmesh.GetNodes()):
    fec = pmesh.GetNodes().OwnFEC()
    delete_fec = False
    if (myid == 0):
        print("Using isoparametric FEs: %s" % fec.Name())
else:
    fec = mfem.H1_FECollection(1, dim)
    delete_fec = True

fespace = mfem.ParFiniteElementSpace(pmesh, fec)

size = fespace.GlobalTrueVSize()
if (myid == 0):
    print("Number of finite element unknowns: %d" % size)


Number of finite element unknowns: 289


In [20]:
# Initiate ROM related variables
max_num_snapshots = 100
update_right_SV = False
isIncremental = False
data_dir = os.getcwd()+"/data/"
baseName = "elliptic_eigenproblem_"
basis_filename = data_dir+baseName+"basis"
solveTimer, assembleTimer, mergeTimer = StopWatch(), StopWatch(), StopWatch()

In [21]:
# 10. Set BasisGenerator if offline
if (offline):
    options = libROM.Options(fespace.GetTrueVSize(), nev, nev,
                            update_right_SV)
    snapshot_basename = data_dir+baseName + "par" + f"{id}"
    generator = libROM.BasisGenerator(options, isIncremental, snapshot_basename)

In [22]:
# 11. The merge phase
if (merge):
    mergeTimer.Start()
    options = libROM.Options(fespace.GetTrueVSize(), max_num_snapshots, nev,
                            update_right_SV)
    generator = libROM.BasisGenerator(options, isIncremental, basis_filename)
    for paramID in range(nsets):
        snapshot_filename = "%s%d_snapshot" % (baseName+"par", paramID)
        generator.loadSamples(data_dir+snapshot_filename,"snapshot", 5)

    generator.endSamples() # save the merged basis file
    mergeTimer.Stop()
    if (myid == 0):
        print("Elapsed time for merging and building ROM basis: %e second\n" %
               mergeTimer.duration)
    del generator
    del options
    MPI.Finalize()

In [23]:
class Conductivity(mfem.PyCoefficient):
    """
    Defines the conductivity as a function of position.
    
    This class extends mfem.PyCoefficient and specifies the conductivity based on the problem and position vector x.
    """

    def __init__(self, problem, amplitude, center, bb_min, bb_max):
        """
        Initializes the Conductivity object.
        
        Parameters:
        - problem (int): The problem setup identifier.
        - amplitude (float): The amplitude value for the conductivity calculation.
        - center (array): The center position as a NumPy array.
        - bb_min (array): The minimum bounding box coordinates as a NumPy array.
        - bb_max (array): The maximum bounding box coordinates as a NumPy array.
        """
        super().__init__()
        self.problem = problem
        self.amplitude = amplitude
        self.center = center
        self.bb_min = bb_min
        self.bb_max = bb_max

    def EvalValue(self, x):
        """
        Evaluates the conductivity at a given position x.
        
        Parameters:
        - x (array): Position at which to evaluate the conductivity, given as a NumPy array.
        
        Returns:
        - float: The evaluated conductivity value at position x.
        """
        cx = 1.0
        if self.problem == 1:
            return 1.0
        elif self.problem == 2:
            cx += self.amplitude
            for i in range(len(x)):
                if 8 * abs(x[i] - self.center[i]) > (self.bb_max[i] - self.bb_min[i]):
                    cx = 1.0
            return cx
        elif self.problem in [3, 4]:
            return 1.0
        return 0.0

In [24]:
class Potential(mfem.PyCoefficient):
    """
    Defines the potential as a function of position.
    
    This class extends mfem.PyCoefficient and specifies the potential based on the problem and position vector x.
    """

    def __init__(self, problem, amplitude, center, h_max):
        """
        Initializes the Potential object.
        
        Parameters:
        - problem (int): The problem setup identifier.
        - amplitude (float): The amplitude value for the potential calculation.
        - center (array): The center position as a NumPy array.
        - h_max (float): The maximum mesh size.
        """
        super().__init__()
        self.problem = problem
        self.amplitude = amplitude
        self.center = center
        self.h_max = h_max


    def EvalValue(self, x):
        """
        Evaluates the potential at a given position x.
        
        Parameters:
        - x (array): Position at which to evaluate the potential, given as a NumPy array.
        
        Returns:
        - float: The evaluated potential value at position x.
        """
        radius = 5.0 * self.h_max
        # x_np = x.GetDataArray()  # Convert MFEM Vector to NumPy array
        # print(type(self.center))
        d_sq = np.sqrt(np.sum((x-self.center.GetDataArray())**2))
        if self.problem in [1, 2]:
            return 0.0
        elif self.problem == 3:
            return self.amplitude * np.exp(-d_sq / np.power(radius, 2.0))
        elif self.problem == 4:
            return self.amplitude * d_sq
        return 0.0

In [25]:
assembleTimer.Start()
one = mfem.ConstantCoefficient(1.0)
# Create a parallel grid function

kappa_0 = Conductivity(problem, amplitude, center, bb_min, bb_max)
v_0 = Potential(problem, amplitude, center, h_max)

In [26]:
c_gf = mfem.ParGridFunction(fespace)
c_gf.ProjectCoefficient(kappa_0)

In [27]:
# 13. Define the solution vector x as a parallel finite element grid function
#     corresponding to fespace. Initialize x with initial guess of zero,
#     which satisfies the boundary conditions.

p_gf = mfem.ParGridFunction(fespace)
p_gf.ProjectCoefficient(v_0)

In [28]:
# 8. Determine the list of true (i.e. parallel conforming) essential
#    boundary dofs. In this example, the boundary conditions are defined
#    by marking all the boundary attributes from the mesh as essential
#    (Dirichlet) and converting them to a list of true dofs.

if (pmesh.bdr_attributes.Size() > 0):
    ess_bdr = mfem.intArray(pmesh.bdr_attributes.Max())
    ess_bdr.Assign(1 if dirichlet else 0)

In [29]:
# 14. Set up the parallel bilinear form a(.,.) on the finite element space
#     corresponding to the Laplacian operator -Delta, by adding the Diffusion
#     domain integrator.

a = mfem.ParBilinearForm(fespace)
a.AddDomainIntegrator(mfem.DiffusionIntegrator(kappa_0))
a.AddDomainIntegrator(mfem.MassIntegrator(v_0))
a.Assemble()
a.EliminateEssentialBCDiag(ess_bdr, 1.0)
a.Finalize()

In [30]:
one = mfem.ConstantCoefficient(1.0)
m = mfem.ParBilinearForm(fespace)
m.AddDomainIntegrator(mfem.MassIntegrator(one))
m.Assemble()
m.EliminateEssentialBCDiag(ess_bdr, 1/np.finfo(np.float64).min)
m.Finalize()

In [32]:
# m.PrintMatlab('DATA_TEST2.m')

In [35]:
A = mfem.HypreParMatrix()
M = mfem.HypreParMatrix()

# Parallel assemble the bilinear forms
A = a.ParallelAssemble()
M = m.ParallelAssemble()

In [36]:
# Assuming `A` is an instance of HypreParMatrix and is already defined.
# Arow = mfem.Operator()
Arow = None

# Check if SuperLU solver is available and selected
try:
    from mfem.common import SuperLURowLocMatrix
    if slu_solver:  # Assuming slu_solver is a boolean variable
        Arow = mfem.SuperLURowLocMatrix(A)
except ImportError:
    pass

# Check if STRUMPACK solver is available and selected
try:
    from mfem.common import STRUMPACKRowLocMatrix
    if sp_solver:  # Assuming sp_solver is a boolean variable
        Arow = mfem.STRUMPACKRowLocMatrix(A)
except ImportError:
    pass

# Stop the assembly timer
assembleTimer.Stop()  # You need to have the `assembleTimer` defined and started somewhere in your code.

# Delete the bilinear forms
del a
del m

# Create a parallel grid function
# x = mfem.ParGridFunction(fespace)
eigenfunction_i = mfem.ParGridFunction(fespace)
eigenvector_i = mfem.Vector(size)
# Create a HypreLOBPCG solver
lobpcg = mfem.HypreLOBPCG(comm)

# Create arrays for eigenvalues and eigenvectors
eigenvalues = mfem.doubleArray()
# eigenvectors = mfem.DenseMatrix()
eigenvectors = []

In [37]:
prescribe_init = False

In [38]:
if fom or offline:
    # Define and configure the LOBPCG eigensolver and the BoomerAMG preconditioner for A
    precond = None

    if not slu_solver and not sp_solver and not cpardiso_solver:
        amg = mfem.HypreBoomerAMG(A)
        amg.SetPrintLevel(verbose_level)
        precond = amg
    else:
        if slu_solver:
            try:
                from mfem import SuperLUSolver
                superlu = mfem.SuperLUSolver(MPI.COMM_WORLD)
                superlu.SetPrintStatistics(verbose_level > 0)
                superlu.SetSymmetricPattern(True)
                superlu.SetColumnPermutation("PARMETIS")
                superlu.SetOperator(Arow)
                precond = superlu
            except ImportError:
                pass

        if sp_solver:
            try:
                from mfem import STRUMPACKSolver
                strumpack = STRUMPACKSolver(MPI.COMM_WORLD)
                strumpack.SetPrintFactorStatistics(True)
                strumpack.SetPrintSolveStatistics(verbose_level > 0)
                strumpack.SetKrylovSolver("DIRECT")
                strumpack.SetReorderingStrategy("METIS")
                strumpack.SetMatching("NONE")
                strumpack.SetCompression("NONE")
                strumpack.SetOperator(Arow)
                strumpack.SetFromCommandLine()
                precond = strumpack
            except ImportError:
                pass

        if cpardiso_solver:
            try:
                from mfem import CPardisoSolver
                cpardiso = CPardisoSolver(A.GetComm())
                cpardiso.SetMatrixType("REAL_STRUCTURE_SYMMETRIC")
                cpardiso.SetPrintLevel(verbose_level)
                cpardiso.SetOperator(A)
                precond = cpardiso
            except ImportError:
                pass

    # Create and configure the HypreLOBPCG solver
    lobpcg = mfem.HypreLOBPCG(MPI.COMM_WORLD)
    lobpcg.SetNumModes(nev)
    lobpcg.SetRandomSeed(seed)
    lobpcg.SetPreconditioner(precond)
    lobpcg.SetMaxIter(lobpcg_niter)
    lobpcg.SetTol(1e-8)
    lobpcg.SetPrecondUsageMode(1)
    lobpcg.SetPrintLevel(verbose_level)
    lobpcg.SetMassMatrix(M)
    lobpcg.SetOperator(A)



    if prescribe_init and (fom or (offline and id > 0)):
        snapshot_vecs = [np.loadtxt(data_dir+f"{baseName}ref_snapshot_{i}.{myid}") for i in range(nev)]
        lobpcg.SetInitialVectors(np.array(snapshot_vecs))
        if myid == 0:
            print("LOBPCG initial vectors set")


    solveTimer.Start()
    lobpcg.Solve()
    solveTimer.Stop()
    
    # Extract eigenvalues
    eigenvalues = mfem.doubleArray(nev)
    lobpcg.GetEigenvalues(eigenvalues)

    for i in range(nev):
        if myid == 0:
            print(f"Eigenvalue {i}: {eigenvalues[i]}")
        if offline:
            eigenfunction_i.Assign(lobpcg.GetEigenvector(i))
            eigenfunction_i /= sqrt(mfem.InnerProduct(eigenfunction_i, eigenfunction_i))
            # generator.takeSample(eigenfunction_i.GetDataArray(),0,0)
            generator.takeSample(eigenfunction_i.GetDataArray())

            if prescribe_init and id == 0:

                snapshot_filename = data_dir+f"{baseName}ref_snapshot_{i}"
                snapshot_vec = lobpcg.GetEigenvector(i)
                np.savetxt(snapshot_filename, snapshot_vec)  # Assuming snapshot_vec is a NumPy array or similar
                if myid == 0:
                    print(f"Saved {snapshot_filename}")


    if offline:
        print("here")
        generator.writeSnapshot()
        del generator
        del options

    # del precond

Eigenvalue 0: 19.802707356800557
Solving generalized eigenvalue problem with preconditioning

block size 4

No constraints



Initial Max. Residual   5.83031250290685e+01


 Num MPI tasks = 1

 Num OpenMP threads = 1


BoomerAMG SETUP PARAMETERS:

 Max levels = 25
 Num levels = 2

 Strength Threshold = 0.250000
 Interpolation Truncation Factor = 0.000000
 Maximum Row Sum Threshold for Dependency Weakening = 0.900000

 Coarsening Type = HMIS 

 No. of levels of aggressive coarsening: 1

 Interpolation on agg. levels= multipass interpolation
 measures are determined locally


 No global partition option chosen.

 Interpolation = extended+i interpolation

Operator Matrix Information:

             nonzero            entries/row          row sums
lev    rows  entries sparse   min  max     avg      min         max
  0     289     1913  0.023     1    9     6.6  -8.327e-16   1.667e+00
  1       9       49  0.605     4    9     5.4   6.661e-16   5.094e+00


Interpolation Matrix Information:
 

In [39]:
if online:
    # 16. Read the reduced basis
    assembleTimer.Start()
    reader = libROM.BasisReader(basis_filename)

    if rdim != -1:
        spatialbasis = reader.getSpatialBasis(rdim)
    else:
        spatialbasis = reader.getSpatialBasis(ef)

    numRowRB = spatialbasis.numRows()
    numColumnRB = spatialbasis.numColumns()
    if myid == 0:
        print(f"Spatial basis dimension is {numRowRB} x {numColumnRB}")

    # 17. Form ROM operator
    ReducedA = libROM.Matrix()
    ComputeCtAB(A, spatialbasis, spatialbasis, ReducedA)
    A_mat = mfem.DenseMatrix(ReducedA.numRows(), ReducedA.numColumns())

    rows, cols = ReducedA.numRows(), ReducedA.numColumns()  # Get matrix dimensions
    ReducedA_data = ReducedA.getData().reshape((rows, cols))  # Reshape the flat data into a matrix form

    # Create a DenseMatrix from the reshaped data
    A_mat.Set(1.0, mfem.DenseMatrix(ReducedA_data))

    ReducedM = libROM.Matrix()
    ComputeCtAB(M, spatialbasis, spatialbasis, ReducedM)
    M_mat = mfem.DenseMatrix(ReducedM.numRows(), ReducedM.numColumns())
    
    rows_M, cols_M = ReducedM.numRows(), ReducedM.numColumns()  # Get matrix dimensions
    ReducedM_data = ReducedM.getData().reshape((rows_M, cols_M))  # Reshape the flat data into a matrix form


    M_mat.Set(1.0, mfem.DenseMatrix(ReducedM_data))
    ### I am here 

    assembleTimer.Stop()

    eigenvalues_rom = mfem.Vector()  # Initialize with correct size
    reduced_eigenvectors = mfem.DenseMatrix()  # Initialize with correct size


    # 18. Solve ROM
    solveTimer.Start()
    A_mat.Eigenvalues(M_mat, eigenvalues_rom, reduced_eigenvectors)
    solveTimer.Stop()

    if myid == 0:

        size = eigenvalues_rom.Size()
        data_ptr = eigenvalues_rom.GetDataArray()
        eigenvalues = mfem.doubleArray(size)

        for i in range(size):
            eigenvalues[i] = data_ptr[i]

        mode_rom = mfem.Vector(numRowRB)
        modes_rom = []

        for i in range(min(eigenvalues_rom.Size(), nev)):
            print(f"Eigenvalue {i}: = {eigenvalues[i]}")

    # tmp = mfem.DenseMatrix(eigenvectors)
    # eigenvectors = mfem.DenseMatrix(nev, numRowRB)

    for j in range(min(eigenvalues_rom.Size(), nev)):
        reduced_eigenvector_j = mfem.Vector()
        reduced_eigenvectors.GetColumn(j, reduced_eigenvector_j)
        reduced_eigenvector_j_carom = libROM.Vector(reduced_eigenvector_j.GetDataArray(), False, False)
        eigenvector_j_carom = spatialbasis.mult(reduced_eigenvector_j_carom)

        mode_rom.Assign(eigenvector_j_carom.getData())

        mode_rom /= sqrt(mfem.InnerProduct(mode_rom,mode_rom))
        modes_rom.append(mfem.Vector(mode_rom))
        
        del eigenvector_j_carom

    sol_eigenvalue_name_fom = data_dir+f"sol_eigenvalues_fom.{myid:06d}"
    eigenvalues_fom = mfem.Vector(nev)
    # Pass file path directly
    eigenvalues_fom.Load(sol_eigenvalue_name_fom,nev)


    # Assuming 'eigenvalues_rom' has been computed already (e.g., another vector of same size)
    diff_eigenvalues = mfem.Vector(nev)

    for i in range(nev):
        diff_eigenvalues[i] = eigenvalues_fom[i] - eigenvalues_rom[i]
        if myid == 0:
            print(f"FOM solution for eigenvalue {i} = {eigenvalues_fom[i]}")
            print(f"ROM solution for eigenvalue {i} = {eigenvalues_rom[i]}")
            print(f"Absolute error of ROM solution for eigenvalue {i} = {abs(diff_eigenvalues[i])}")
            print(f"Relative error of ROM solution for eigenvalue {i} = {abs(diff_eigenvalues[i]) / abs(eigenvalues_fom[i])}")


    # Load and calculate errors for eigenvectors
    eigenvectors = []
    for i in range(nev):
        mode_name_fom = f"eigenfunction_fom_{i:02d}.{myid:06d}"
        
        # Load FOM eigenvector from file
        mode_fom = mfem.Vector(numRowRB)
        mode_fom.Load(data_dir+mode_name_fom, numRowRB)

        eigenvector_i = mfem.Vector(numRowRB)  # Initialize eigenvector_i to zero
        eigenvector_i.Assign(0.0)
        
        for j in range(nev):
            if myid == 0 and verbose_level > 0:
                print(f"correlation_matrix({j+1},{i+1}) = {mfem.InnerProduct(mode_fom, modes_rom[j])};")
            
            # If eigenvalues are nearly identical, add the corresponding ROM mode
            if abs(eigenvalues_fom[j] - eigenvalues_fom[i]) < 1e-6:
                eigenvector_i.Add(mfem.InnerProduct(mode_fom, modes_rom[j]), modes_rom[j])

        # Store the computed eigenvector
        eigenvectors.append(eigenvector_i)
        
        # Compute the difference between FOM and ROM eigenvectors
        mode_fom.Add(-1.0, eigenvector_i)
        diff_norm = sqrt(mfem.InnerProduct(mode_fom, mode_fom))
        
        if myid == 0:
            print(f"Relative l2 error of ROM eigenvector {i} = {diff_norm}")



    del spatialbasis
    del A_mat
    del M_mat

In [40]:
# Define the filename format helper
def get_filename(prefix, myid, digits=6):
    return f"{prefix}{myid:0{digits}}"


## Print Mesh
mesh_name = get_filename(data_dir+"elliptic_eigenproblem-mesh.", myid)
pmesh.Print(mesh_name, precision)


## Print Eigenvalue
if fom or offline:
    sol_eigenvalue_name = get_filename(data_dir+"sol_eigenvalues_fom.", myid)

if online:
    sol_eigenvalue_name = get_filename(data_dir+"sol_eigenvalues.", myid)

# pmesh.Print(sol_eigenvalue_name, precision)

In [41]:
# 19. Save the refined mesh and the modes in parallel.
sign_eigenvectors = mfem.Vector(nev) ## ?? no idea what is this!

mode_prefix =  data_dir+"eigenfunction_"

if fom or offline:
    mode_prefix +="fom_"

elif online:
    mode_prefix += "rom_"


# Save the eigenvalues
with open(sol_eigenvalue_name, 'w') as eigvals:
    for val in eigenvalues:
        eigvals.write(f"{val}\n")


# Save the eigenfunctions
for i in range(nev):

    if fom or offline:
        eigenfunction_i = lobpcg.GetEigenvector(i)
        eigenfunction_i /= sqrt(mfem.InnerProduct(eigenfunction_i, eigenfunction_i))
    else:
        eigenfunction_i = eigenvectors[i]

    # mode_ref = mfem.Vector(eigenvector_i.Size())
    mode_name = f"{mode_prefix}{i:02}.{myid:06}"


    with open(mode_name, 'w') as mode_ofs:
        for val in eigenfunction_i:
            mode_ofs.write(f"{val}\n")

In [42]:
# if online:
#     # Initialize FOM solution
#     eigenvalues_fom = mfem.Vector(nev)

#     eigenvalues_fom.Load(sol_eigenvalue_name_fom, eigenvalues_fom.Size())

#     diff_eigenvalues = mfem.Vector(nev)
#     for i in range(min(eigenvalues.Size(), nev)):
#         diff_eigenvalues[i] = eigenvalues_fom[i] - eigenvalues[i]
#         if myid == 0:
#             print(f"FOM solution for eigenvalue {i} = {eigenvalues_fom[i]}")
#             print(f"ROM solution for eigenvalue {i} = {eigenvalues[i]}")
#             print(f"Absolute error of ROM solution for eigenvalue {i} = {abs(diff_eigenvalues[i])}")
#             print(f"Relative error of ROM solution for eigenvalue {i} = {abs(diff_eigenvalues[i]) / abs(eigenvalues_fom[i])}")

#     # Calculate errors of eigenvectors
#     for i in range(min(eigenvalues.Size(), nev)):
#         mode_name_fom = get_filename("eigenfunction_fom_", i, myid, 2)
#         mode_fom = mfem.Vector(eigenvectors.NumCols())
#         mode_fom.Load(mode_name_fom, eigenvectors.NumCols())

#         fomNorm = np.sqrt(mfem.InnerProduct(mode_fom, mode_fom))

#         for j in range(min(eigenvalues.Size(), nev)):
#             if abs(eigenvalues_fom[j] - eigenvalues_fom[i]) < 1e-6:
#                 mode_name = get_filename("mode_rom_", j, myid, 2)
#                 mode_rom = mfem.Vector(eigenvectors.NumCols())
#                 mode_rom.Load(mode_name, eigenvectors.NumCols())

#                 mode_fom.Add(-mfem.InnerProduct(mode_fom, mode_rom), mode_rom)

#         diffNorm = np.sqrt(mfem.InnerProduct(mode_fom, mode_fom))
#         if myid == 0:
#             print(f"Relative l2 error of ROM eigenvector {i} = {diffNorm / fomNorm}")


##### Visualization

In [43]:
visit_dc = None
if visit:
    if offline:
        visit_dc = mfem.VisItDataCollection(f"{baseName}offline_par{id}", pmesh)
    elif fom:
        visit_dc = mfem.VisItDataCollection(f"{baseName}fom", pmesh)
    elif online:
        visit_dc = mfem.VisItDataCollection(f"{baseName}rom", pmesh)
    
    visit_dc.RegisterField("Conductivity", c_gf)
    visit_dc.RegisterField("Potential", p_gf)
    visit_eigenvectors = []

    for i in range(min(nev, eigenvalues.Size())):
        if fom or offline:
            eigenvector_i = lobpcg.GetEigenvector(i)
        else:
            eigenvector_i = mfem.Vector()
            eigenvectors.GetRow(i, eigenvector_i)

        eigenvector_i *= sign_eigenvectors[i] / np.sqrt(mfem.InnerProduct(eigenvector_i, eigenvector_i))

        x.Assign(eigenvector_i)
        eigenvector_gf = mfem.ParGridFunction(x)
        visit_eigenvectors.append(mfem.Vector(eigenvector_gf))
        visit_dc.RegisterField(f"Eigenmode_{i}", eigenvector_gf)

    visit_dc.SetCycle(0)
    visit_dc.SetTime(0.0)
    visit_dc.Save()

    visit_eigenvectors.clear()


sout = None
if visualization:
    vishost = "localhost"
    visport = 19916
    sout = mfem.socketstream(vishost, visport)
    sout << f"parallel {num_procs} {myid}\n"
    
    good = sout.good()
    all_good = MPI.COMM_WORLD.allreduce(good, op=MPI.MIN)
    
    if not all_good:
        sout.close()
        visualization = False
        if myid == 0:
            print(f"Unable to connect to GLVis server at {vishost}:{visport}")
            print("GLVis visualization disabled.")
    else:
        for i in range(min(nev, eigenvalues.Size())):
            if myid == 0:
                print(f"Eigenmode {i + 1}/{nev}, Lambda = {eigenvalues[i]}")

            if fom or offline:
                eigenvector_i = lobpcg.GetEigenvector(i)
            else:
                eigenvector_i = mfem.Vector()
                eigenvectors.GetRow(i, eigenvector_i)
            
            eigenvector_i *= sign_eigenvectors[i] / np.sqrt(mfem.InnerProduct(eigenvector_i, eigenvector_i))
            x.Assign(eigenvector_i)

            sout << f"parallel {num_procs} {myid}\nsolution\n" << pmesh << x << "\n"
            sout << f"window_title 'Eigenmode {i + 1}/{nev}, Lambda = {eigenvalues[i]}'\n"

            if myid == 0:
                c = input("press (q)uit or (c)ontinue --> ").strip()
            else:
                c = None
            c = MPI.COMM_WORLD.bcast(c, root=0)

            if c != 'c':
                break
        sout.close()


In [44]:
# 20. Print timing info
if myid == 0:
    if fom or offline:
        print(f"Elapsed time for assembling FOM: {assembleTimer.duration:e} seconds")
        print(f"Elapsed time for solving FOM: {solveTimer.duration:e} seconds")
    if online:
        print(f"Elapsed time for assembling ROM: {assembleTimer.duration:e} seconds")
        print(f"Elapsed time for solving ROM: {solveTimer.duration:e} seconds")

# 21. Free the used memory.
if fom or offline:
    del lobpcg

del M
del A

del fespace
if order > 0:
    del fec
del pmesh

# Free Arow if it was used
if 'Arow' in locals():
    del Arow

MPI.Finalize()

Elapsed time for assembling FOM: 1.120543e-01 seconds
Elapsed time for solving FOM: 1.201572e-01 seconds
